# 🧠 ค่าความถูกต้อง (Accuracy) และความขัดแย้งของค่าความถูกต้อง (Accuracy Paradox)

ยินดีต้อนรับสู่โน้ตบุ๊กประกอบการอธิบายเรื่อง **ค่าความถูกต้อง (Accuracy)**! ในโน้ตบุ๊กนี้เราจะ:
1. ทำความรู้จักองค์ประกอบของตัวจำแนกคลาสแบบสองกลุ่ม (Binary Classification): True Positives (TP), True Negatives (TN), False Positives (FP) และ False Negatives (FN)
2. เขียนฟังก์ชันคำนวณค่าความถูกต้อง (Accuracy Metric) จากศูนย์ด้วย NumPy พร้อมทดสอบความเที่ยงตรงเทียบกับ `scikit-learn`
3. สาธิตปรากฏการณ์ **ความขัดแย้งของค่าความถูกต้อง (Accuracy Paradox)** โดยการสร้างชุดข้อมูลจำลองที่คลาสไม่สมดุลกันอย่างรุนแรง (เช่น การตรวจจับรอยรั่วท่อแก๊สที่ 99% ของตัวอย่างข้อมูลไม่มีการรั่วไหล)
4. แสดงกรณีตัวอย่างว่าแบบจำลองที่ทายเพียงแค่คลาสส่วนใหญ่ (Majority-class classifier) ที่ไม่มีประโยชน์เลย กลับได้ค่าความถูกต้องที่สูงลิ่วแต่ล้มเหลวในการตรวจจับเป้าหมายจริงโดยสิ้นเชิง
5. เชื่อมโยงแนวคิดเหล่านี้เพื่อตอบคำถามว่า เหตุใดระบบตรวจจับวัตถุอย่าง YOLO จึงไม่ได้ใช้ค่าความถูกต้อง (Accuracy) ทั่วไปในการวัดผล

เริ่มต้นด้วยการนำเข้าไลบรารีที่จำเป็นกันก่อนครับ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import accuracy_score

# กำหนดค่า seed เพื่อให้ได้ผลลัพธ์การสุ่มเหมือนเดิมทุกครั้ง
np.random.seed(42)

## 1. การคำนวณค่าความถูกต้องจากพื้นฐาน (Calculating Accuracy from Scratch)

เรามาลองเขียนฟังก์ชันเพื่อนับจำนวนของ TP, TN, FP, FN และรวบรวมข้อมูลเหล่านั้นกลับมาเป็นค่าความถูกต้อง (Accuracy) กันครับ

In [ ]:
def calculate_outcomes(y_true, y_pred):
    """
    คำนวณค่า True Positives, True Negatives, False Positives และ False Negatives
    """
    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    return TP, TN, FP, FN

def custom_accuracy(y_true, y_pred):
    TP, TN, FP, FN = calculate_outcomes(y_true, y_pred)
    total = TP + TN + FP + FN
    if total == 0:
        return 0.0
    return (TP + TN) / total

# กำหนดอาร์เรย์ตัวอย่างจำลองสำหรับการเปรียบเทียบ (เช่น การตรวจสอบคลาสของ Bounding Box)
y_true = np.array([1, 0, 1, 1, 0, 1, 0, 0, 1, 0])
y_pred = np.array([1, 0, 1, 0, 0, 1, 1, 0, 1, 0])

TP, TN, FP, FN = calculate_outcomes(y_true, y_pred)
acc_scratch = custom_accuracy(y_true, y_pred)
acc_sklearn = accuracy_score(y_true, y_pred)

print("--- Outcome Stats ---")
print(f"TP: {TP} | TN: {TN} | FP: {FP} | FN: {FN}")
print(f"Custom Accuracy: {acc_scratch:.4f}")
print(f"Sklearn Accuracy: {acc_sklearn:.4f}")

## 2. ความขัดแย้งของค่าความถูกต้อง (The Accuracy Paradox จากปัญหา Class Imbalance)

เพื่อเข้าใจว่าเหตุใดค่าความถูกต้อง (Accuracy) จึงนำพาเราไปสู่การประเมินที่คลาดเคลื่อนได้ เราจะลองจำลองชุดข้อมูลการหารอยรั่วบนท่อส่งแก๊สขึ้นมา:
-   จำนวนตัวอย่างท่อทั้งหมด = 1,000 จุด
-   จุดที่มีการรั่วไหลจริง (คลาส 1) = 10 จุด (คิดเป็น 1% ของชุดข้อมูล)
-   จุดปกติที่ไม่มีรอยรั่ว (คลาส 0) = 990 จุด (คิดเป็น 99% ของชุดข้อมูล)

เราจะลองทดสอบแบบจำลองคัดกรอง 2 ตัว:
1.  **Dumb Classifier (แบบจำลองไร้สมอง):** ทำการเดาคำตอบส่งๆ ว่า "ปกติ" (คลาส 0) เสมอไม่ว่าจะเจอท่อแบบใด
2.  **Sensible Classifier (แบบจำลองที่มีเหตุผล):** แบบจำลองจริงที่ผ่านกระบวนการเรียนรู้ ซึ่งอาจมีการทายที่ถูกต้องและผิดพลาดคละเคล้ากันไป

In [ ]:
# สร้างป้ายกำกับเฉลยจริงที่มีคลาสไม่สมดุล
y_true_imbalanced = np.zeros(1000)
# สุ่มสร้างรอยรั่วจริง 10 รอย (คลาส 1) ลงในดัชนีแบบสุ่ม
leak_indices = np.random.choice(1000, 10, replace=False)
y_true_imbalanced[leak_indices] = 1

# 1. ผลลัพธ์ของแบบจำลองไร้สมอง (ทาย 0 หรือปกติทั้งหมด)
y_pred_dumb = np.zeros(1000)

# 2. ผลลัพธ์ของแบบจำลองที่มีเหตุผล
y_pred_sensible = np.zeros(1000)
# ค้นพบรอยรั่วจริงได้อย่างถูกต้อง 8 จุด
correct_leaks_detected = leak_indices[:8]
y_pred_sensible[correct_leaks_detected] = 1
# สุ่มสร้างสัญญาณเตือนภัยเท็จ (False Positives) 15 จุดบนท่อที่ไม่มีรอยรั่ว
non_leak_indices = np.where(y_true_imbalanced == 0)[0]
false_alarms = np.random.choice(non_leak_indices, 15, replace=False)
y_pred_sensible[false_alarms] = 1

# คำนวณค่าชี้วัด
acc_dumb = custom_accuracy(y_true_imbalanced, y_pred_dumb)
acc_sensible = custom_accuracy(y_true_imbalanced, y_pred_sensible)

tp_dumb, tn_dumb, fp_dumb, fn_dumb = calculate_outcomes(y_true_imbalanced, y_pred_dumb)
tp_sensible, tn_sensible, fp_sensible, fn_sensible = calculate_outcomes(y_true_imbalanced, y_pred_sensible)

# แสดงตารางตารางเปรียบเทียบผล
df_compare = pd.DataFrame({
    'Metric': ['Accuracy', 'True Positives (Leaks Found)', 'False Negatives (Missed Leaks)', 'False Positives (False Alarms)'],
    'Dumb Model': [f"{acc_dumb*100:.2f}%", tp_dumb, fn_dumb, fp_dumb],
    'Sensible Model': [f"{acc_sensible*100:.2f}%", tp_sensible, fn_sensible, fp_sensible]
})

print(df_compare.to_string(index=False))

สังเกตผลลัพธ์ในตารางเทียบกันครับ!
-   **Dumb Model (โมเดลไร้สมอง)** ได้ค่าความถูกต้องสูงถึง **99.00%** แต่กลับไม่สามารถระบุรอยรั่วได้เลยแม้แต่จุดเดียว (0 True Positives, ปล่อยรั่วหนีรอดไป 10 จุด)
-   **Sensible Model (โมเดลมีเหตุผล)** ได้ค่าความถูกต้องรองลงมาเล็กน้อยคือ **97.70%** แต่กลับตรวจจับจุดรั่วจริงได้ถึง 8 ใน 10 จุด (8 True Positives, หลุดรอดไปเพียง 2 จุดเท่านั้น)
นี่คือปรากฏการณ์ **ความขัดแย้งของค่าความถูกต้อง (Accuracy Paradox)** ซึ่งชี้ให้เห็นว่าในสถานการณ์จริง แบบจำลองที่มีเปอร์เซ็นต์ความถูกต้องต่ำกว่าเล็กน้อย กลับเป็นตัวที่เราอยากนำไปติดตั้งใช้งานจริงมากกว่า!

In [ ]:
# พล็อตกราฟเปรียบเทียบผลลัพธ์การนับ
plt.figure(figsize=(10, 5))
x = np.arange(2)
width = 0.35

plt.bar(x - width/2, [tp_dumb, tp_sensible], width, label='True Leaks Detected (TP)', color='green')
plt.bar(x + width/2, [fn_dumb, fn_sensible], width, label='Missed Leaks (FN)', color='red')

plt.xticks(x, ['Dumb Model', 'Sensible Model'])
plt.ylabel('Count')
plt.title('Leak Detection Comparison: Dumb vs. Sensible Model')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

## 💡 ความเชื่อมโยงสู่ Computer Vision และ YOLO
ทำไมเราจึงไม่นำค่าความถูกต้องของการจัดคลาสทั่วไปมาประเมินประสิทธิภาพของระบบตรวจจับวัตถุอย่าง YOLO?
1.  **ค่าลบจริงที่เป็นอนันต์ (Infinite True Negatives):** บนเฟรมภาพถ่ายหนึ่งใบ มีกรอบสี่เหลี่ยมจำนวนเป็นอนันต์ที่ *ไม่ปรากฏวัตถุ* อยู่ภายใน หากโมเดลตรวจจับวาดกล่อง Bounding Box สำเร็จได้อย่างถูกต้อง 2 กล่อง โมเดลทำ True Negatives ( TN หรือการตรวจเจอความไม่มีวัตถุ) ไปเท่าไหร่? คำตอบคือ โมเดลได้ละเว้นกรอบสี่เหลี่ยมอื่นในภาพไปหลายล้านพิกเซล ซึ่งหากเรานำค่า $TN = \infty$ ไปแทนลงในสูตรความถูกต้องดั้งเดิม:
    $$\text{Accuracy} = \frac{TP + \infty}{TP + \infty + FP + FN} = 1.0 = 100\%$$
    ค่าความถูกต้องจะกลายเป็น 100% เสมอ ส่งผลให้ค่าวัด Accuracy นี้ไม่มีประโยชน์เลยสำหรับประเมินระบบงานตรวจจับวัตถุ
2.  **การประเมินความแม่นยำด้านพิกัดตำแหน่ง (Evaluating Localization):** ค่าความถูกต้องไม่สามารถคำนวณได้ว่ากรอบสี่เหลี่ยมที่โมเดลวาดขึ้นมานั้นมีความกระชับครอบคลุมพอดีกับกรอบเฉลยจริงเท่าใด ดังนั้น วงการคอมพิวเตอร์วิทัศน์จึงเลือกประเมินด้วยค่า **Intersection over Union (IoU)** เพื่อวิเคราะห์ความแม่นยำของขอบกล่อง และคำนวณดัชนีเฉลี่ยความถูกต้อง **mean Average Precision (mAP)** ซึ่งละเว้นการนำเอาค่า True Negatives (TN) เข้ามาร่วมหารในการวัดผลการแยกคลาสวัตถุ